In [1]:
import arinc424

In [2]:
# Parse the whole CIFP file
import sys
from pathlib import Path

import ipynbname

project_root = Path(ipynbname.path()).parent.parent.parent.parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

cifp_path = project_root / "data" / "cifp" / "FAACIFP18.txt"
if not cifp_path.exists():
    raise Exception(f"CIFP File could not be located at {cifp_path.absolute()}")
else:
    print(f"CIFP File found: {cifp_path.absolute()}")

CIFP File found: /Volumes/CrucialX/project-rustlingtree/data/cifp/FAACIFP18.txt


In [3]:
# Goal: to build two pandas dataframes, one containing the waypoints
# For example: `SUSAP KDFWK4CZUBOV K40    W     N33304871W097540488                       E0033     NAR           ZUBOV                    698362403` contains the waypoint information
# And the other containing the procedure instructions
# For example: `SUSAP KDFWK4EJOVEM64BELFR 010BELFRK4PC0E       IF                                             18000                        711462407
# SUSAP KDFWK4EJOVEM64BELFR 020LETNNK4PC0E       TF                                 B FL300FL240     290                     711472407
# SUSAP KDFWK4EJOVEM64BELFR 030ZUBOVK4PC0Ex       TF                                 B FL230FL200                             711482407
# SUSAP KDFWK4EJOVEM64BELFR 040VKTRYK4PC0EE      TF                                 B FL18015000     280                     711492407
# `

# Algorithm: 1. Filter all rows with filter_str = "SUSAP KDFW*" (wildcard must be supported)
# 2. You get all the rows like SUSAP ... call the arinc424 library to parse the record (silently if possible). 
# 3. In the transition dataframe, you must note down the fields: section_code, sid_star_approach_identifier, route_type, transition_identifier, sequence_number, fix_identifier, icao_code, section_code_2, altitude, altitude_2
# 4. Find all involved waypoints (from transition identifier, fix identifier) and add them to the fixes dataframe with geographical altitudes

In [4]:
from IPython.display import display
import pandas as pd

from scenario.cifp_parser.airport_related_extractor import build_airport_procedure_dataframes

airport_icao = "KDFW"
output_dir = project_root / "data" / "kdfw_procs"
report_path = output_dir / "airport_related_report.txt"
transitions_df, fixes_df, summary = build_airport_procedure_dataframes(
    cifp_path,
    airport_icao,
    report_path=report_path,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print(f"Airport: {airport_icao}")
print(f"Filter hits: {summary.filter_hit_count}")
print(f"Processed rows: {summary.processed_row_count}")
print(f"Fixes added: {summary.fixes_added_count}")
print(f"Missing fix references: {summary.unresolved_fix_reference_count}")
print(f"Missing fix identifiers: {summary.unresolved_fix_identifier_count}")
print(f"Filter parse failures: {summary.filter_parse_failure_count}")
print(f"Deduplicated fix rows skipped: {summary.deduplicated_fix_row_count}")

print(f"Transitions: {len(transitions_df)} rows")
display(transitions_df.head(20))

print(f"Fixes: {len(fixes_df)} rows")
display(fixes_df.head(20))

print("\nProcedure legs for one example transition:")
display(transitions_df.loc[transitions_df["sid_star_approach_identifier"] == "JOVEM6"])

print("\nFix lookup sample:")
display(fixes_df.loc[fixes_df["identifier"].isin(["BELFR", "SPERA", "RW17C", "TTT", "GEEKY", "ABI"])].sort_values(["identifier", "fix_type"]))

# Save the airport related procedures to external CSVs
output_dir.mkdir(parents=True, exist_ok=True)

transitions_csv = output_dir / "airport_related.csv"
fixes_csv = output_dir / "airport_related_fixes.csv"

transitions_df.to_csv(transitions_csv, index=False)
fixes_df.to_csv(fixes_csv, index=False)

print(f"Wrote {report_path}")
print(f"Wrote {transitions_csv}")
print(f"Wrote {fixes_csv}")


Airport: KDFW
Filter hits: 2430
Processed rows: 2127
Fixes added: 425
Missing fix references: 228
Missing fix identifiers: 8
Filter parse failures: 1
Deduplicated fix rows skipped: 0
Transitions: 2127 rows


/var/folders/r4/52yfj2854bl5v3zcbd28w4_r0000gn/T/ipykernel_54599/1576107884.py:9: RuntimeWarning: Skipped 1 procedure candidate records that the ARINC-424 parser could not read.
  transitions_df, fixes_df, summary = build_airport_procedure_dataframes(
/var/folders/r4/52yfj2854bl5v3zcbd28w4_r0000gn/T/ipykernel_54599/1576107884.py:9: RuntimeWarning: Unresolved procedure references for KDFW: ALL, KDFW, RW13B, RW17B, RW18B, RW31B, RW35B, RW36B
  transitions_df, fixes_df, summary = build_airport_procedure_dataframes(


,section_code,sid_star_approach_identifier,route_type,transition_identifier,sequence_number,fix_identifier,icao_code,section_code_2,altitude,altitude_2
0,PD,AKUNA9,4,RW17C,010,NaN,NaN,NaN,NaN,NaN
1,PD,AKUNA9,4,RW17C,020,SPERA,K4,PC,NaN,NaN
2,PD,AKUNA9,4,RW17C,030,JGIRL,K4,PC,05000,NaN
3,PD,AKUNA9,4,RW17C,040,CORTS,K4,PC,NaN,NaN
4,PD,AKUNA9,4,RW17C,050,BIGGD,K4,PC,NaN,NaN
5,PD,AKUNA9,4,RW17C,060,CMORE,K4,PC,NaN,NaN
6,PD,AKUNA9,4,RW17C,070,AKUNA,K4,EA,NaN,NaN
7,PD,AKUNA9,4,RW17R,010,NaN,NaN,NaN,NaN,NaN
8,PD,AKUNA9,4,RW17R,020,SPERA,K4,PC,NaN,NaN
9,PD,AKUNA9,4,RW17R,030,JGIRL,K4,PC,05000,NaN


Fixes: 425 rows


,identifier,fix_type,source_section_code,airport_icao,latitude_raw,longitude_raw,elevation_raw,latitude_deg,longitude_deg,elevation_ft
0,RW17C,runway,PG,KDFW,N32545654,W097013351,00562,32.915706,-97.025975,562.0
1,SPERA,waypoint,PC,KDFW,N32465775,W096594205,NaN,32.782708,-96.995014,NaN
2,JGIRL,waypoint,PC,KDFW,N32455436,W096562163,NaN,32.765100,-96.939342,NaN
3,CORTS,waypoint,PC,KDFW,N32460765,W096495359,NaN,32.768792,-96.831553,NaN
4,BIGGD,waypoint,PC,KDFW,N32513159,W096483310,NaN,32.858775,-96.809194,NaN
5,CMORE,waypoint,PC,KDFW,N32591666,W096483249,NaN,32.987961,-96.809025,NaN
6,AKUNA,waypoint,EA,NaN,N33270248,W096492330,NaN,33.450689,-96.823139,NaN
7,RW17R,runway,PG,KDFW,N32545660,W097014758,00567,32.915722,-97.029883,567.0
8,RW18L,runway,PG,KDFW,N32545688,W097030265,00602,32.915800,-97.050736,602.0
9,BPARK,waypoint,PC,KDFW,N32463176,W097045151,NaN,32.775489,-97.080975,NaN



Procedure legs for one example transition:


,section_code,sid_star_approach_identifier,route_type,transition_identifier,sequence_number,fix_identifier,icao_code,section_code_2,altitude,altitude_2
1308,PE,JOVEM6,4,BELFR,010,BELFR,K4,PC,NaN,NaN
1309,PE,JOVEM6,4,BELFR,020,LETNN,K4,PC,FL300,FL240
1310,PE,JOVEM6,4,BELFR,040,VKTRY,K4,PC,FL180,15000
1311,PE,JOVEM6,4,FAWNT,010,FAWNT,K4,EA,NaN,NaN
1312,PE,JOVEM6,4,FAWNT,020,WNSTR,K4,PC,FL280,FL240
...,...,...,...,...,...,...,...,...,...,...
1367,PE,JOVEM6,5,ALL,030,MSSLE,K4,PC,12000,NaN
1368,PE,JOVEM6,5,ALL,040,SILER,K4,PC,12000,NaN
1369,PE,JOVEM6,5,ALL,050,KBOOM,K4,PC,NaN,NaN
1370,PE,JOVEM6,5,ALL,060,FUEWL,K4,PC,NaN,NaN



Fix lookup sample:


,identifier,fix_type,source_section_code,airport_icao,latitude_raw,longitude_raw,elevation_raw,latitude_deg,longitude_deg,elevation_ft
73,ABI,vor,D,NaN,N32285279,W099514843,NaN,32.481331,-99.863453,NaN
262,BELFR,waypoint,PC,KDFW,N33533935,W098402606,NaN,33.894264,-98.673906,NaN
182,GEEKY,waypoint,EA,NaN,N32164632,W099530109,NaN,32.279533,-99.883636,NaN
0,RW17C,runway,PG,KDFW,N32545654,W097013351,00562,32.915706,-97.025975,562.0
1,SPERA,waypoint,PC,KDFW,N32465775,W096594205,NaN,32.782708,-96.995014,NaN
72,TTT,vor,D,NaN,N32520898,W097022581,NaN,32.869161,-97.040503,NaN


Wrote /Volumes/CrucialX/project-rustlingtree/data/kdfw_procs/airport_related_report.txt
Wrote /Volumes/CrucialX/project-rustlingtree/data/kdfw_procs/airport_related.csv
Wrote /Volumes/CrucialX/project-rustlingtree/data/kdfw_procs/airport_related_fixes.csv
